# Inspect operating assumptions offline

Use the installed development revision of `epsm_data` with Python 3.11+. No Core, database, network or engine is needed. Values retain Sveby/BEN source identities and Atemp reference bases. This example writes no files and demonstrates selection and portability, not simulation or regulatory compliance.

In [ ]:
from epsm_data import (
    OperatingProfile, available_operating_profiles,
    select_operating_profile, select_dhw_profile,
)

profiles = available_operating_profiles()
assert "sveby_school_v1" in profiles
residential = select_operating_profile("multi_family")
print(residential)
assert residential.values["use"]["dhw_kwh_m2_year"] == 21

The published office and school profiles need caller calendars and a compatible load reference area. Selecting a minimum airflow supplies no installed HVAC system. No unsupported building use automatically selects a generic fallback.

In [ ]:
office = select_operating_profile("office", version="1")
assert "calendar" in office.values["requirements"]
assert office.source_references
print(office.values["requirements"])

BEN is explicitly selected separately. Its historical EPC coefficient is useful thermal DHW per square metre of Atemp per year, before production efficiency, and excludes storage/circulation losses.

In [ ]:
ben = select_dhw_profile("multi_family", purpose="epc_normal_use")
assert ben.values["dhw_kwh_m2_year"] == 25
assert residential.values["use"]["dhw_kwh_m2_year"] == 21
assert ben.values["area_basis"] == "Atemp"

A detached exchange dictionary can be edited without changing the definition. Custom profile input uses the same portable class; Core performs later physical validation.

In [ ]:
import json

restored = OperatingProfile.from_dict(json.loads(json.dumps(office.to_dict())))
assert restored == office
custom = OperatingProfile(
    "example.partial-office",
    {"building_assumptions": {"use": {"occupant_density_per_m2": 0.04}}},
    source_references=("example:declared-assumption",),
    notes=("Partial demonstration; remaining operation inputs are unresolved.",),
)
edited = custom.with_changes(notes=("Independent authored copy.",))
assert custom.notes != edited.notes
print(custom)